# 06 — Experiment 4: + Self-Supervised Pretraining (SSL)
Pretrains the CNN backbone on **unlabeled** MRI images using SimCLR-style contrastive learning
(NT-Xent loss over two augmented views per image), then transplants the learned backbone
weights into a fresh Experiment-3 architecture before supervised fine-tuning.

**[our contribution]** — SSL pretraining. Architecture is identical to Experiment 3; only the
initial weights differ. This tests whether self-supervised pretraining improves data efficiency/
generalization given how few labeled OASIS images there are per class.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from src import config as cfg
from src.utils import set_seed, check_environment
from src.data_loader import get_unlabeled_dataset_for_ssl
from src.models import ssl_pretrain as ssl

set_seed()
check_environment()

In [ ]:
# Pretraining step — uses only images, no labels
unlabeled_ds = get_unlabeled_dataset_for_ssl()
pretrained_backbone, ssl_history = ssl.pretrain_ssl(unlabeled_ds)

In [ ]:
import matplotlib.pyplot as plt
plt.plot(ssl_history)
plt.xlabel('epoch'); plt.ylabel('NT-Xent loss'); plt.title('SSL pretraining loss')
plt.show()

In [ ]:
# Fine-tuning step: build Experiment 3 architecture, transplant SSL weights, then train supervised
from src.train import train_one_experiment

model, history, metrics = train_one_experiment('cbam_transformer_ssl')

In [ ]:
import numpy as np
from src.utils import plot_confusion_matrix

y_true = np.load(os.path.join(cfg.RESULTS_DIR, 'cbam_transformer_ssl_y_true.npy'))
y_pred = np.load(os.path.join(cfg.RESULTS_DIR, 'cbam_transformer_ssl_y_pred.npy'))
plot_confusion_matrix(y_true, y_pred, title='Experiment 4: + SSL Pretraining')

Now all four experiments have saved metrics — proceed to notebook 08 for the ablation comparison, or notebook 07 for explainability on this final model.